In [1]:
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import numpy as np
from pathlib import Path

In [28]:
window_size = 51 # In meters, will be converted to pixels based on raster resolution
temp_file = Path('../data/traversals/asheville/pm/trav.shp')
nlcd_file = Path('../data/nlcd/landcover_asheville.tif')
greenspace_file = Path('../data/0.6m-Greenspace_mapping/Asheville__NC_resultv2.tif')
ndvi_albedo_file = Path('../data/sentinel/Asheville_NDVI_Albedo.tif')
elev_file = Path('../data/elevation/elevation_asheville_2023-07-24.tif')

In [29]:
gdf = gpd.read_file(temp_file)
obj = [{'temp': v.item()} for v in gdf['t_f'].values]

coords = [[g.geometry.x, g.geometry.y] for _, g in gdf.to_crs("EPSG:5070").iterrows()]
coords = np.array(coords)

for i, o in enumerate(obj):
    o['coords'] = coords[i, :].tolist()

In [30]:
# Process NLCD data

_NLCD_CLASSES = [11, 12, 21, 22, 23, 24, 31, 41, 42, 43, 51, 52, 71, 72, 73, 74, 81, 82, 90, 95]

with rasterio.open(nlcd_file) as src:
    # Reproject points to raster CRS if needed
    if gdf.crs != src.crs:
        gdf_reproj = gdf.to_crs(src.crs)
    else:
        gdf_reproj = gdf

    for idx, point in gdf_reproj.iterrows():
        # Get pixel coordinates
        row, col = src.index(point.geometry.x, point.geometry.y)
        
        # Calculate window around the point
        half_window = window_size // 2
        window = Window(
            col - half_window,
            row - half_window,
            window_size,
            window_size
        )
        
        try:
            # Read the window
            data = src.read(1, window=window)
            
            # Handle nodata values
            if src.nodata is not None:
                data = np.ma.masked_equal(data, src.nodata)
            
            obj[idx]['nlcd_window'] = data.tolist()
        except Exception as e:
            # Handle edge cases (points outside raster, etc.)
            obj[idx]['nlcd_window'] = None
            print(f"Warning: Could not extract value for point {idx}: {e}")



In [31]:
# Greenspace file
with rasterio.open(greenspace_file) as src:
    # Reproject points to raster CRS if needed
    if gdf.crs != src.crs:
        gdf_reproj = gdf.to_crs(src.crs)
    else:
        gdf_reproj = gdf


    for idx, point in gdf_reproj.iterrows():
        # Get pixel coordinates
        row, col = src.index(point.geometry.x, point.geometry.y)
        
        # Calculate window around the point
        half_window = window_size // 2
        window = Window(
            col - half_window,
            row - half_window,
            window_size,
            window_size
        )
        
        try:
            # Read the window
            data = src.read(1, window=window)
            
            # Handle nodata values
            # if src.nodata is not None:
            #     data = np.ma.masked_equal(data, src.nodata)

            obj[idx]['greenspace_window'] = data.tolist()    
        except Exception as e:
            # Handle edge cases (points outside raster, etc.)
            obj[idx]['greenspace_window'] = None
            print(f"Warning: Could not extract value for point {idx}: {e}")


In [32]:
# NDVI / ALBEDO
with rasterio.open(ndvi_albedo_file) as src:
    # Reproject points to raster CRS if needed
    print(src.res)
    if gdf.crs != src.crs:
        gdf_reproj = gdf.to_crs(src.crs)
    else:
        gdf_reproj = gdf


    for idx, point in gdf_reproj.iterrows():
        # Get pixel coordinates
        row, col = src.index(point.geometry.x, point.geometry.y)
        
        # Calculate window around the point
        
        half_window = window_size // 2
        window = Window(
            col - half_window,
            row - half_window,
            window_size,
            window_size
        )
        
        try:
            # Read the window
            data = src.read((1, 2), window=window)
            
            obj[idx]['ndvi_albedo_window'] = data.tolist()
        except Exception as e:
            # Handle edge cases (points outside raster, etc.)
            obj[idx]['ndvi_albedo_window'] = None
            print(f"Warning: Could not extract value for point {idx}: {e}")


(8.983152841195215e-05, 8.983152841195215e-05)


In [36]:
with rasterio.open(elev_file) as src:
    # Reproject points to raster CRS if needed
    if gdf.crs != src.crs:
        gdf_reproj = gdf.to_crs(src.crs)
    else:
        gdf_reproj = gdf


    coords = [(point.geometry.x, point.geometry.y) for _, point in gdf_reproj.iterrows()]
    
    # Sample all points at once (more efficient)
    elevations = list(src.sample(coords, 1))

    for idx, elev in enumerate(elevations):
        obj[idx]['elev'] = elev[0].item()
    # for idx, point in gdf_reproj.iterrows():
    #     # Read elevation point
    #     row, col = src.index(point.geometry.x, point.geometry.y)
    #     elev = src.read(1, window=Window(col, row, 1, 1))[0, 0]
    #     obj[idx]['elev'] = elev.item()


In [37]:
# Filter out points with masked arrays
filtered_obj = []
for item in obj:
    if 'nlcd_window' in item and item['nlcd_window'] is not None and np.array(item['nlcd_window']).shape != (window_size, window_size):
        continue
    if 'greenspace_window' in item and item['greenspace_window'] is not None and np.array(item['greenspace_window']).shape != (window_size, window_size):
        continue
    if 'ndvi_albedo_window' in item and item['ndvi_albedo_window'] is not None and np.array(item['ndvi_albedo_window']).shape != (2, window_size, window_size):
        continue
    filtered_obj.append(item)

In [39]:
item['elev']

656

In [40]:
import json
# obj = {'data': filtered_obj}


for i in range(len(filtered_obj)):
    with open(f'../asheville/{i}.json', 'w') as f:
        json.dump(filtered_obj[i], f)
# with open('../filtered_data.json', 'w') as f:
#     json.dump(obj, f)